# Round 8: config-mix averaging, value-count features, forest + FE - S6E10

Round 7 showed 4 extra seeds add no measured gain over 3 (not proof averaging is maxed) and that
ExtraTrees is much weaker than the booster here, so a 50/50 family blend dilutes. The evidence
points at DIVERSITY WITHIN the strong family and at untried feature information. Three bounded
tests, existing data only:

- **Exp 26: config-mix averaging.** Average the OOF probabilities of four near-tied plateau
  LightGBM configs (incumbent, lr=0.03, num_leaves=127, stochastic-seed-42). Hypothesis: configs
  that tie on average may still disagree row-by-row more than reseeds of one config do - if so,
  mixing them buys what extra seeds did not.
- **Exp 27: value-count features (FE for the booster).** For every raw column, add the per-fold
  TRAIN-ONLY frequency of the exact value (21 new columns, validation values unseen in fold-train
  map to 0). Mechanism, stated honestly: this synthetic data was generated from a smaller original
  survey, and third-party public work CLAIMS exact-value multiplicity carries signal - those are
  third-party claims, not our results; this is our measured test, and it uses only competition data.
- **Exp 28: value-count features for the forest (FE for all models, RF preference).** The same 21
  count features added to the round-7 ExtraTrees. FE has not helped LightGBM yet; whether it helps
  the forest family is a separate, untested question.

Bar change this round: Exp 22 (0.95878) is the reference candidate, so the refit guard now compares
against Exp 22's round-6 fold constants, not the round-3 incumbent. Cross-run pairing caveat:
every config involved has reproduced its folds EXACTLY across runs so far (incumbent 3x, Exp 22
once in-run) - the caveat is stated, not assumed away.


In [ ]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # current best candidate (bar)
E24_FOLDS_R7 = [0.95634, 0.95523, 0.95656, 0.95625, 0.95625]        # round-7 ExtraTrees reference
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)


In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file():
    if Path('playground-series-s6e10/data/train.csv').is_file():
        competition_dir = Path('playground-series-s6e10')
    elif Path('../data/train.csv').is_file():
        competition_dir = Path('..')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def to_sklearn(train_part, other_parts=()):
    num_cols = [c for c in train_part.columns if c not in CATEGORICAL]
    medians = train_part[num_cols].median()
    def prep(df):
        out = df.copy()
        out[num_cols] = out[num_cols].fillna(medians)
        return pd.get_dummies(out, columns=CATEGORICAL)
    ref = prep(train_part)
    cols = ref.columns
    return ref, [prep(p).reindex(columns=cols, fill_value=0) for p in other_parts]

def add_counts(train_part, other_parts=()):
    # per-column exact-value frequencies computed on train_part only; unseen values -> 0
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, **params):
    frame = to_lgbm(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
        # use whichever signature the installed version supports (no warning either way).
        if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
            model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')


## Reference: incumbent recompute

Same-run incumbent (Exp 8 config) with the determinism check; its OOF is also reused as a member of
the Exp 26 mix (no extra compute).


In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc
INC_MEAN = float(np.mean(aucs_inc))
print(f'  incumbent mean OOF AUC: {INC_MEAN:.5f} +/- {np.std(aucs_inc):.5f} | pooled: {roc_auc_score(target, oof_inc):.6f}')


## Exp 26: config-mix averaging (4 plateau LightGBM configs)

Mean of OOF probabilities from: incumbent (lr=0.05/leaves=63), lr=0.03, num_leaves=127, and the
stochastic config (seed 42). All four tied within noise of each other singly; the hypothesis is
row-level disagreement between configs exceeds what reseeding one config explores.


In [ ]:
print('Exp 26 members:')
oof_lr03, aucs_lr03, iters_lr03 = oof_lgbm(features, target, 'E26-lr03', learning_rate=0.03, n_estimators=3000)
print(f'  lr=0.03 mean: {np.mean(aucs_lr03):.5f}')
oof_l127, aucs_l127, iters_l127 = oof_lgbm(features, target, 'E26-l127', num_leaves=127)
print(f'  leaves=127 mean: {np.mean(aucs_l127):.5f}')
oof_st42, aucs_st42, iters_st42 = oof_lgbm(features, target, 'E26-st42', seed=42, **STOCH)
print(f'  stochastic s42 mean: {np.mean(aucs_st42):.5f}')

oof26 = np.mean([oof_inc, oof_lr03, oof_l127, oof_st42], axis=0)
aucs26 = fold_aucs(oof26, target)
for fold, a in enumerate(aucs26, 1):
    print(f'  E26 fold {fold} (4-config mix): AUC {a:.5f}')
print(f'Exp 26 mean OOF AUC: {np.mean(aucs26):.5f} +/- {np.std(aucs26):.5f} | pooled: {roc_auc_score(target, oof26):.6f}')


## Exp 27: value-count features for the booster

21 new columns: the fold-TRAIN-only exact-value frequency of every raw column. One change on top of
the incumbent config. Leakage control: counts are computed inside each fold from that fold's
training rows only; validation values never seen in fold-train map to 0.


In [ ]:
print('Exp 27: incumbent config + 21 value-count features (fold-local counts)')
oof27 = np.zeros(len(features))
aucs27, iters27 = [], []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(features, target), 1):
    X_tr, y_tr = features.iloc[tr_idx], target.iloc[tr_idx]
    X_va, y_va = features.iloc[va_idx], target.iloc[va_idx]
    X_tr_c, (X_va_c,) = add_counts(X_tr, [X_va])
    X_tr_c, X_va_c = to_lgbm(X_tr_c), to_lgbm(X_va_c)
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_tr_c, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
    model = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
    # use whichever signature the installed version supports (no warning either way).
    if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
        model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    else:
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    oof27[va_idx] = model.predict_proba(X_va_c)[:, 1]
    a = roc_auc_score(y_va, oof27[va_idx])
    aucs27.append(a); iters27.append(model.best_iteration_)
    print(f'  E27 fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
print(f'Exp 27 mean OOF AUC: {np.mean(aucs27):.5f} +/- {np.std(aucs27):.5f} | pooled: {roc_auc_score(target, oof27):.6f}')


## Exp 28: value-count features for the forest (FE for all models)

The same fold-local count features added to the round-7 ExtraTrees config (400 trees, leaf=5,
median-impute + one-hot). FE has not helped LightGBM so far; the forest family is a separate
measured question, and Andrea asked for FE work across model families.


In [ ]:
print('Exp 28: ExtraTrees 400/leaf5 + 21 value-count features (fold-local counts)')
oof28 = np.zeros(len(features))
aucs28 = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(features, target), 1):
    X_tr, y_tr = features.iloc[tr_idx], target.iloc[tr_idx]
    X_va, y_va = features.iloc[va_idx], target.iloc[va_idx]
    X_tr_c, (X_va_c,) = add_counts(X_tr, [X_va])
    X_tr_p, (X_va_p,) = to_sklearn(X_tr_c, [X_va_c])
    et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1,
                              random_state=RANDOM_STATE)
    et.fit(X_tr_p, y_tr)
    oof28[va_idx] = et.predict_proba(X_va_p)[:, 1]
    a = roc_auc_score(y_va, oof28[va_idx])
    aucs28.append(a)
    print(f'  E28 fold {fold}: AUC {a:.5f}')
print(f'Exp 28 mean OOF AUC: {np.mean(aucs28):.5f} +/- {np.std(aucs28):.5f} | pooled: {roc_auc_score(target, oof28):.6f}')


## Results, verdicts, and refit guard

Two pairings per candidate: vs the in-run incumbent (continuity with earlier rounds) and vs Exp
22's round-6 fold constants - the current best candidate and the bar for any refit. Cross-run
pairing caveat applies as stated in the intro. A candidate earns a refit ONLY by beating the Exp
22 bar under the paired band; clearing the old incumbent is no longer sufficient.


In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

results = {}
for exp_id, name, aucs, oof, second_ref in [
    ('26', 'LGBM 4-config mix', aucs26, oof26, None),
    ('27', 'LGBM + value-count features', aucs27, oof27, None),
    ('28', 'ExtraTrees + value-count features', aucs28, oof28, ('round-7 Exp 24 (ExtraTrees)', E24_FOLDS_R7)),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs, INC_FOLDS)
    d22, m22, s22, lab22 = paired_verdict(aucs, E22_FOLDS_R6)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label,
                           m22=m22, lab22=lab22)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.6f}')
    print(f'  paired deltas vs in-run incumbent: ' + ' '.join(f'{x:+.5f}' for x in deltas) + f' (mean {mean_d:+.5f}) | {label}')
    print(f'  paired deltas vs Exp 22 (the bar): ' + ' '.join(f'{x:+.5f}' for x in d22) + f' (mean {m22:+.5f}) | {lab22}')
    if second_ref:
        d3, m3, s3, lab3 = paired_verdict(aucs, second_ref[1])
        print(f'  paired vs {second_ref[0]}: mean {m3:+.5f} | {lab3}')
print()

best_oof = max([0.95878] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Best OOF mean incl. Exp 22 bar: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

winners = [k for k, r in results.items()
           if r['m22'] > 0 and 'WORKING SIGNAL: better' in r['lab22']]
if not winners:
    print('No candidate beat the Exp 22 bar: Exp 22 (submission_round6.csv, local on Mac) remains the preferred submission candidate; no submission file written this round.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    print(f'Winner vs the Exp 22 bar: Exp {win} ({results[win]["name"]}) - refits on all rows below; logged as a working signal, exploratory.')


In [ ]:
# Refit ONLY fires on a whole-data working-signal winner over the Exp 22 bar.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    def full_lgbm(n_est, feat=features, feat_test=None, seed=RANDOM_STATE, **params):
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=n_est,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        m = lgb.LGBMClassifier(**cfg)
        m.fit(to_lgbm(feat), target)
        return m.predict_proba(to_lgbm(feat_test if feat_test is not None else test_features))[:, 1]
    if win == '26':
        # each member refit on all rows with its own in-run fold-mean best_iteration basis
        test_probabilities = np.mean([
            full_lgbm(int(np.mean(iters_inc))),
            full_lgbm(int(np.mean(iters_lr03)), learning_rate=0.03),
            full_lgbm(int(np.mean(iters_l127)), num_leaves=127),
            full_lgbm(int(np.mean(iters_st42)), seed=42, **STOCH),
        ], axis=0)
    elif win == '27':
        train_c, (test_c,) = add_counts(features, [test_features])
        test_probabilities = full_lgbm(int(np.mean(iters27)), feat=train_c, feat_test=test_c)
    else:  # '28'
        train_c, (test_c,) = add_counts(features, [test_features])
        X_all, (X_test_p,) = to_sklearn(train_c, [test_c])
        et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1,
                                  random_state=RANDOM_STATE)
        et.fit(X_all, target)
        test_probabilities = et.predict_proba(X_test_p)[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round8.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round8_oof.npz',
         oof_inc=np.asarray(oof_inc), aucs_inc=np.asarray(aucs_inc), iters_inc=np.asarray(iters_inc),
         oof26=np.asarray(oof26), aucs26=np.asarray(aucs26),
         oof_lr03=np.asarray(oof_lr03), aucs_lr03=np.asarray(aucs_lr03), iters_lr03=np.asarray(iters_lr03),
         oof_l127=np.asarray(oof_l127), aucs_l127=np.asarray(aucs_l127), iters_l127=np.asarray(iters_l127),
         oof_st42=np.asarray(oof_st42), aucs_st42=np.asarray(aucs_st42), iters_st42=np.asarray(iters_st42),
         oof27=np.asarray(oof27), aucs27=np.asarray(aucs27), iters27=np.asarray(iters27),
         oof28=np.asarray(oof28), aucs28=np.asarray(aucs28),
         y=np.asarray(target).astype(int))
meta = {
    'round': 8,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold; value counts fold-train-only; ExtraTrees median-impute + one-hot per fold',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'incumbent': 'LightGBM lr=0.05, num_leaves=63, n_estimators=1000, early stop 50 (Exp 8 config, recomputed in-run)',
        'exp26': 'mean OOF of 4 configs: incumbent; lr=0.03 n_est=3000; num_leaves=127; stochastic(bagging 0.9/freq1, feature_fraction 0.9) seed 42',
        'exp27': 'incumbent config + 21 fold-train-only exact-value count features',
        'exp28': 'ExtraTreesClassifier n_estimators=400, min_samples_leaf=5 + same 21 count features; per-fold median impute + one-hot',
    },
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round8_oof.json').write_text(_json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


## What happens next

- If Exp 26's config mix beats the Exp 22 bar: diversity within the strong family is the lever -
  follow up with weighted mixes (weights chosen on a FRESH re-partition, never on the selection
  OOF) or adding the mix to the 3-seed averaging.
- If Exp 27 wins: value-multiplicity is real signal on this data - follow-ups are smoothed counts,
  count-based ratios, and adding counts to the stochastic-averaged config (the current best).
- If Exp 28 wins while 27 does not: FE belongs to the forest family here - invest in forest-side
  features (counts, delay handling) rather than booster-side.
- If everything is null: tested approaches still have not reached 0.96 on this data alone (finite
  configs, not proof it cannot be done); the missing-Arrival-Delay experiment remains queued, and
  the external original-survey dataset stays gated on Andrea's explicit OK.
